# Space-map — Check the supplied section order

Compare the supplied section order with a similarity-based suggested order, and flag dissimilar sections before 3D reconstruction.

This notebook computes pairwise SIFT feature-match similarities, displays a similarity heatmap, and reports the supplied order, a suggested order, and dissimilar sections. The suggested order is a heuristic for inspection.

**Data:** set `DATA` to your coordinate CSV. Run all cells — no registration
needed, this works directly from raw coordinates.

**Processed data:** [Supplementary Dataset 1](https://static-content.springer.com/esm/art%3A10.1038%2Fs41592-026-03310-8/MediaObjects/41592_2026_3310_MOESM8_ESM.zip). **Raw data:** [source repository](https://humantumoratlas.org/publications/hta10_2026__rongduo-han?tab=10x-xenium-iss).


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

# --- edit this to your local file ---
DATA = 'data/xenium_polyp.csv.gz'
X_COL, Y_COL, LAYER_COL = 'x', 'y', 'layer'
N_LAYERS = None   # limit layers for a quick check; None = all
OUT = 'out/section_order'

## Compute pairwise SIFT similarity and check order

`check_order_from_csv` builds temporary slices, renders density images,
counts SIFT feature matches between every pair, and returns the similarity
matrix plus a suggested section order and any flagged outliers.

In [ ]:
import space_map

os.makedirs(OUT, exist_ok=True)
result = space_map.qc.check_order_from_csv(
    DATA,
    x_col=X_COL, y_col=Y_COL, layer_col=LAYER_COL,
    n_layers=N_LAYERS,
    plot=True,
    save_plot=f'{OUT}/section_similarity.png',
)
print('input order  :', result['ids'])
print('suggested    :', result['suggested_order'])
print('flagged      :', result['flagged'] or '(none)')

## Per-section best-match counts

Each bar is the strongest match that section has to any other. Low similarity
can reflect morphology, sampling, or data quality; it does not prove damage or
incorrect acquisition order.

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'section': result['ids'],
    'best_match': result['best_match'].astype(int),
    'flagged': [s in result['flagged'] for s in result['ids']],
})
print(summary.to_string(index=False))

## What to do next

- **Order looks wrong?** Compare the suggestion with acquisition records and
  tissue morphology before choosing the section sequence.
- **Flagged sections?** Inspect their morphology and metadata before deciding
  whether exclusion is justified. A candidate filtered list can be formed with:
  ```python
  keep_ids = [s for s in result['ids'] if s not in result['flagged']]
  ```
- **Save the matrix** for further analysis:
  ```python
  import numpy as np
  np.save(f'{OUT}/similarity_matrix.npy', result['matrix'])
  ```